# 06 — Quality Evaluation (Phase E / Batch A)

**Scope of this notebook (Batch A only):**
1. Source-location → chunk-index mapping (the §11 E.1 "ground truth" step that was supposed to land in Phase B but didn't — built retroactively here).
2. **§11 E.1** — Recall@k stratified by question type, computed for every Phase D configuration.
3. **§11 E.2** — Q13 multi-doc strict vs. lenient recall.

**Out of scope** (later Phase E batches):
- §11 E.3 atomic-fact rubric grading + Cohen's kappa (Saheed/Eze handoff).
- §11 E.4 cost-quality plots (Anandhu handoff).

**Locked methodological clarification (declared in the Phase E close-out entry).** Recall@k is computed over the **deduplicated unique** `(config_hash, top_k, question_id)` cells. Under temperature=0 + seed=42 + deterministic chunking + CPU sentence-transformers (§13.5), the 3 reps per cell produce byte-identical retrieved_chunk_ids; computing recall@k once per cell is therefore mathematically equivalent to computing per repetition and then averaging. This converts the §11 DoD's nominal 1248 grading rows to ~169 unique answers, a methodological clarification rather than a deviation. Verified by an explicit assert in the sanity-checks cell.

**k values.** §11 reports recall@k for the `top_k` values logged in `query_log.csv`: **{1, 3, 5}**. `top_k=10` was dropped in Phase D Exp3 under §14 risk-register precedent and is therefore absent from `query_log.csv`.

**Cell layout** (mirrors `02_measurement_layer.ipynb` / Phase D notebooks):
1. (this) intro
2. driver — paths and small constants
3. setup — imports, seeds, helpers
4. page-offset table — per-doc PageOffset records
5. source-location parsing — per-question pages_by_doc, sanity-print
6. recall@k computation — load query_log, dedup, compute hits, save `results/recall_log.csv`
7. stratified summary — by config / by question_type / Q13 strict-vs-lenient
8. winner detail — `aadb0cb9` / `top_k=3`
9. sanity asserts (schema lock + value-level)
10. wrap-up


In [1]:
# Cell 2 — driver constants. Edit here, not deeper in the notebook.
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR  = PROJECT_ROOT / "dataset"
PDF_DIR      = DATASET_DIR / "pdfs"
CORPUS_DIR   = PROJECT_ROOT / "corpus_text"
INDICES_DIR  = PROJECT_ROOT / "indices"
RESULTS_DIR  = PROJECT_ROOT / "results"
EVAL_QUESTIONS_PATH = DATASET_DIR / "eval_questions.md"
QUERY_LOG_PATH      = RESULTS_DIR / "query_log.csv"
INDEXING_LOG_PATH   = RESULTS_DIR / "indexing_log.csv"
RECALL_LOG_PATH     = RESULTS_DIR / "recall_log.csv"

# Phase E reports recall@k for every k value present in query_log.csv. The
# project's logged set is {1, 3, 5} — see PROGRESS_LOG.md Phase D / Exp3
# entry for the §14-sanctioned top_k=10 drop.
EXPECTED_K_VALUES = {1, 3, 5}

# Phase E DoD nominally specified 1248 grading rows (16 configs × 13 q × 6
# rep grading slots, 2 graders). With the locked dedup rule below, the
# expected unique-cell count is the number of distinct (config_hash,
# top_k, question_id) triples in query_log.csv. The Batch-A task brief
# estimated ~169; the actual count after dedup is 156 because the cell
# (aadb0cb9, top_k=3) appears in Phase D Exp1, Exp2 (rebuild), and Exp3
# (reuse path), all of which collapse into a single deduplicated cell
# rather than 3. The assert in cell 9 fails loudly on any drift.
EXPECTED_DEDUP_CELL_COUNT = 156

RANDOM_SEED = 42  # §15

for p in (PDF_DIR, CORPUS_DIR, INDICES_DIR, RESULTS_DIR, EVAL_QUESTIONS_PATH,
          QUERY_LOG_PATH, INDEXING_LOG_PATH):
    assert p.exists(), f"Missing required path: {p}"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"recall_log will be written to: {RECALL_LOG_PATH}")


PROJECT_ROOT: <project_root>
recall_log will be written to: <project_root>/results/recall_log.csv


In [2]:
# Cell 3 — setup: imports, seeds, lightweight helpers.
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.metrics import set_global_seeds
from src.eval_questions import load_eval_questions
from src.chunking import load_chunks
from src.recall import (
    PDF_TO_SHORT_NAME,
    EXPECTED_RECALL_COLS,
    Q13_ID,
    LIST_SEP,
    build_page_offsets,
    parse_source_pages,
    gold_ranges_for_question,
    gold_chunk_ids,
    build_recall_rows,
    recall_row_dict,
    assert_recall_schema,
)

set_global_seeds(RANDOM_SEED)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
print("Setup complete. Determinism guard set (seed=42).")


Setup complete. Determinism guard set (seed=42).


In [3]:
# Cell 4 — page-offset table per source doc.
#
# Each PDF is re-extracted with pypdf, cleaned identically to Phase B
# (`clean_extracted_text`), and located in `corpus_text/<doc>.txt` via the
# whitespace-stripped fingerprint matcher in `src/recall.py`. Result: a
# `PageOffset(page, start_char, end_char)` per PDF page.
page_offsets_by_doc: dict[str, list] = {}
for pdf_filename, short_name in PDF_TO_SHORT_NAME.items():
    pdf_path = PDF_DIR / pdf_filename
    corpus_text = (CORPUS_DIR / f"{short_name}.txt").read_text(encoding="utf-8")
    page_offsets_by_doc[short_name] = build_page_offsets(pdf_path, corpus_text)

page_summary = pd.DataFrame([
    {
        "source_doc": doc,
        "n_pages": len(offsets),
        "n_mapped": sum(1 for po in offsets if po.start_char is not None),
        "n_empty": sum(1 for po in offsets if po.start_char is None),
        "corpus_chars": (CORPUS_DIR / f"{doc}.txt").stat().st_size,
    }
    for doc, offsets in page_offsets_by_doc.items()
])
print(page_summary.to_string(index=False))


   source_doc  n_pages  n_mapped  n_empty  corpus_chars
     01_BLOOM       15        15        0         44839
       02_IEA      304       304        0        779376
    03_Google      120       120        0        382819
      04_EPRI       35        35        0        100619
05_Greenpeace       27        27        0         51804


In [4]:
# Cell 5 — source-location parsing → per-question gold pages.
#
# Print every question's parsed (doc -> sorted page list) for visual
# inspection. Q13 is the only multi-doc case; its source_location is
# segmented on ";" so each segment's pages attach to the named doc.
questions = load_eval_questions(EVAL_QUESTIONS_PATH)
questions_by_id = {q.question_id: q for q in questions}
print(f"Loaded {len(questions)} evaluation questions.\n")

rows = []
for q in questions:
    pages = parse_source_pages(q)
    pages_view = {d: sorted(p) for d, p in pages.items()}
    rows.append({
        "question_id": q.question_id,
        "type": q.question_type,
        "source_doc": q.source_doc,
        "pages_by_doc": pages_view,
    })
print(pd.DataFrame(rows).to_string(index=False))


Loaded 13 evaluation questions.

question_id      type             source_doc                                                                      pages_by_doc
        Q01   factoid               01_BLOOM                                                                 {'01_BLOOM': [7]}
        Q02   factoid               01_BLOOM                                                             {'01_BLOOM': [9, 10]}
        Q03 synthesis               01_BLOOM                                                              {'01_BLOOM': [6, 7]}
        Q04 numerical                 02_IEA                                                              {'02_IEA': [13, 14]}
        Q05   factoid                 02_IEA                                                              {'02_IEA': [21, 22]}
        Q06 synthesis                 02_IEA                                                          {'02_IEA': [14, 15, 16]}
        Q07   factoid              03_Google                                  

In [5]:
# Cell 6 — recall@k computation.
#
# Steps:
#   1. Load query_log.csv. Project state: 559 rows × 18 cols (Phase D close).
#   2. Deduplicate to one row per (config_hash, top_k, question_id) cell —
#      the locked methodological clarification (cell 1 / cell 9 assert).
#   3. Load chunks per config_hash from indices/<hash>/chunks.parquet.
#   4. For each unique cell, compute gold_chunk_ids and the per-row hit /
#      strict_hit / lenient_hit fields.
#   5. Persist to results/recall_log.csv with the §11 schema in cell 9.
query_log = pd.read_csv(QUERY_LOG_PATH)
indexing_log = pd.read_csv(INDEXING_LOG_PATH)
print(f"query_log:    {len(query_log)} rows × {len(query_log.columns)} cols")
print(f"indexing_log: {len(indexing_log)} rows × {len(indexing_log.columns)} cols")

dedup = (
    query_log
    .sort_values(["config_hash", "top_k", "question_id", "run_id"])
    .drop_duplicates(subset=["config_hash", "top_k", "question_id"], keep="first")
    .reset_index(drop=True)
)
print(f"deduplicated: {len(dedup)} unique (config_hash, top_k, question_id) cells")

# Determinism check: byte-identical retrieved_chunk_ids across reps for
# every duplicated cell. Any drift would silently bias recall, so we fail
# loudly here.
drift_rows = []
for keys, grp in query_log.groupby(["config_hash", "top_k", "question_id"]):
    if grp["retrieved_chunk_ids"].nunique() > 1:
        drift_rows.append({"keys": keys, "n_unique": grp["retrieved_chunk_ids"].nunique()})
assert not drift_rows, (
    f"Determinism violated: {len(drift_rows)} cells have differing retrieved_chunk_ids "
    f"across reps. Sample: {drift_rows[:3]}"
)
print("Determinism check: 0 mismatches across reps. Dedup is lossless.")

# Load chunks per config_hash from the persisted indices.
config_hashes = sorted(query_log["config_hash"].unique())
chunks_by_config: dict[str, dict[str, list]] = {}
for cfg in config_hashes:
    chunks = load_chunks(INDICES_DIR / cfg / "chunks.parquet")
    by_doc: dict[str, list] = {}
    for c in chunks:
        by_doc.setdefault(c.source_doc, []).append(c)
    chunks_by_config[cfg] = by_doc
print(f"Loaded chunks for {len(chunks_by_config)} config(s): {list(chunks_by_config)}")

# Build the recall_log dataframe.
recall_rows = build_recall_rows(
    dedup.to_dict(orient="records"),
    questions_by_id,
    chunks_by_config,
    page_offsets_by_doc,
)
recall_df = pd.DataFrame([recall_row_dict(r) for r in recall_rows])
recall_df = recall_df[EXPECTED_RECALL_COLS]
assert_recall_schema(recall_df)
RECALL_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
recall_df.to_csv(RECALL_LOG_PATH, index=False)
print(f"Wrote {len(recall_df)} rows to {RECALL_LOG_PATH.relative_to(PROJECT_ROOT)}")
recall_df.head(8)


query_log:    559 rows × 18 cols
indexing_log: 15 rows × 16 cols
deduplicated: 156 unique (config_hash, top_k, question_id) cells
Determinism check: 0 mismatches across reps. Dedup is lossless.
Loaded chunks for 10 config(s): ['1fbb3d49', '6e091273', '6fb92835', '7dffb047', 'a9e9b27f', 'aadb0cb9', 'b1843394', 'b2ee97fc', 'bc360218', 'd7129fb0']
Wrote 156 rows to results/recall_log.csv


,config_hash,top_k,question_id,question_type,source_doc,n_gold_chunks,gold_chunk_ids,retrieved_chunk_ids,hit,strict_hit,lenient_hit,notes
0,1fbb3d49,3,Q01,factoid,01_BLOOM,2,01_BLOOM_0004;01_BLOOM_0005,01_BLOOM_0002;01_BLOOM_0005;01_BLOOM_0000,True,,,
1,1fbb3d49,3,Q02,factoid,01_BLOOM,2,01_BLOOM_0006;01_BLOOM_0007,01_BLOOM_0007;01_BLOOM_0008;01_BLOOM_0002,True,,,
2,1fbb3d49,3,Q03,synthesis,01_BLOOM,3,01_BLOOM_0003;01_BLOOM_0004;01_BLOOM_0005,01_BLOOM_0005;01_BLOOM_0004;01_BLOOM_0007,True,,,
3,1fbb3d49,3,Q04,numerical,02_IEA,2,02_IEA_0006;02_IEA_0007,02_IEA_0036;02_IEA_0022;02_IEA_0037,False,,,
4,1fbb3d49,3,Q05,factoid,02_IEA,2,02_IEA_0011;02_IEA_0012,02_IEA_0013;02_IEA_0015;02_IEA_0147,False,,,
5,1fbb3d49,3,Q06,synthesis,02_IEA,3,02_IEA_0006;02_IEA_0007;02_IEA_0008,02_IEA_0142;02_IEA_0067;04_EPRI_0003,False,,,
6,1fbb3d49,3,Q07,factoid,03_Google,4,03_Google_0000;03_Google_0001;03_Google_0008;0...,03_Google_0039;03_Google_0089;03_Google_0001,True,,,
7,1fbb3d49,3,Q08,factoid,03_Google,4,03_Google_0003;03_Google_0004;03_Google_0006;0...,03_Google_0089;04_EPRI_0012;04_EPRI_0010,False,,,


In [6]:
# Cell 7 — stratified summary across configs.
#
# Three reports:
#   (a) Per-config overall recall@k (k=1,3,5 as available).
#   (b) By-question-type breakdown for the project winner aadb0cb9 / k=3.
#   (c) By-question-type breakdown across all configs at top_k=3.

# (a) Per-config overall recall.
per_config = (
    recall_df
    .groupby(["config_hash", "top_k"], as_index=False)
    .agg(n=("hit", "size"), n_hit=("hit", "sum"))
)
per_config["recall"] = (per_config["n_hit"] / per_config["n"]).round(3)

# Annotate with chunk_size, chunk_overlap, embedding_model from indexing_log.
idx_meta = (
    indexing_log
    .drop_duplicates("config_hash")
    [["config_hash", "chunk_size", "chunk_overlap", "embedding_model"]]
)
per_config = per_config.merge(idx_meta, on="config_hash", how="left")
per_config = per_config.sort_values(["top_k", "recall"], ascending=[True, False])
print("=== (a) Per-config overall recall@k (Phase D configurations) ===")
print(per_config[[
    "config_hash", "top_k", "chunk_size", "chunk_overlap",
    "embedding_model", "n_hit", "n", "recall",
]].to_string(index=False))

# (b) By question type, all configs at top_k=3.
k3 = recall_df[recall_df.top_k == 3].copy()
by_type = (
    k3.groupby(["config_hash", "question_type"], as_index=False)
      .agg(n=("hit", "size"), n_hit=("hit", "sum"))
)
by_type["recall"] = (by_type["n_hit"] / by_type["n"]).round(3)
pivot = by_type.pivot(index="config_hash", columns="question_type", values="recall")
print("\n=== (c) By-question-type recall@3, all configs ===")
print(pivot.fillna(0.0).to_string())


=== (a) Per-config overall recall@k (Phase D configurations) ===
config_hash  top_k  chunk_size  chunk_overlap                         embedding_model  n_hit  n  recall
   aadb0cb9      1         512             20  sentence-transformers/all-MiniLM-L6-v2      6 13   0.462
   6e091273      3         256             10  sentence-transformers/all-MiniLM-L6-v2      9 13   0.692
   aadb0cb9      3         512             20  sentence-transformers/all-MiniLM-L6-v2      9 13   0.692
   b2ee97fc      3         256             20  sentence-transformers/all-MiniLM-L6-v2      9 13   0.692
   a9e9b27f      3         512              0  sentence-transformers/all-MiniLM-L6-v2      8 13   0.615
   b1843394      3        1024             20  sentence-transformers/all-MiniLM-L6-v2      8 13   0.615
   bc360218      3         256              0  sentence-transformers/all-MiniLM-L6-v2      8 13   0.615
   d7129fb0      3         512             20 sentence-transformers/all-mpnet-base-v2      8 13   0.615

In [7]:
# Cell 8 — winner detail (aadb0cb9 / top_k=3) and Q13 strict-vs-lenient.
WINNER_CFG = "aadb0cb9"
WINNER_K = 3

winner = (
    recall_df[(recall_df.config_hash == WINNER_CFG) & (recall_df.top_k == WINNER_K)]
    .sort_values("question_id")
    .reset_index(drop=True)
)
print(f"=== Winner config: {WINNER_CFG} / top_k={WINNER_K} (n={len(winner)}) ===")
for _, r in winner.iterrows():
    suffix = ""
    if r["question_id"] == Q13_ID:
        suffix = f"  [strict={r['strict_hit']}, lenient={r['lenient_hit']}]"
    print(
        f"  {r['question_id']}  ({r['question_type']:>10}, {r['source_doc']:<22})  "
        f"hit={str(bool(r['hit'])):<5}  retrieved={r['retrieved_chunk_ids']}{suffix}"
    )
winner_recall = winner["hit"].mean()
print(f"\nOverall recall@{WINNER_K}: {winner_recall:.3f} ({int(winner['hit'].sum())}/{len(winner)})")
print("By question type:")
for qt, g in winner.groupby("question_type"):
    print(f"  {qt:<10}: {g['hit'].mean():.3f} ({int(g['hit'].sum())}/{len(g)})")

# Q13 strict vs lenient gap, every (config, top_k) pair.
q13 = recall_df[recall_df.question_id == Q13_ID].copy()
q13["strict_bool"] = q13["strict_hit"].map({"True": True, "False": False})
q13["lenient_bool"] = q13["lenient_hit"].map({"True": True, "False": False})
q13_summary = q13[["config_hash", "top_k", "strict_bool", "lenient_bool"]].copy()
q13_summary.columns = ["config_hash", "top_k", "strict_hit", "lenient_hit"]
print("\n=== Q13 strict vs lenient (per config, top_k) ===")
print(q13_summary.sort_values(["top_k", "config_hash"]).to_string(index=False))
n_strict_hits = int(q13_summary["strict_hit"].sum())
n_lenient_hits = int(q13_summary["lenient_hit"].sum())
n_total_q13 = len(q13_summary)
print(
    f"\nQ13 strict hits: {n_strict_hits}/{n_total_q13}  |  "
    f"lenient hits: {n_lenient_hits}/{n_total_q13}  |  "
    f"strict-vs-lenient gap: {n_lenient_hits - n_strict_hits}"
)


=== Winner config: aadb0cb9 / top_k=3 (n=13) ===
  Q01  (   factoid, 01_BLOOM              )  hit=True   retrieved=01_BLOOM_0011;01_BLOOM_0006;01_BLOOM_0017
  Q02  (   factoid, 01_BLOOM              )  hit=True   retrieved=01_BLOOM_0016;01_BLOOM_0006;01_BLOOM_0018
  Q03  ( synthesis, 01_BLOOM              )  hit=True   retrieved=01_BLOOM_0008;01_BLOOM_0010;01_BLOOM_0011
  Q04  ( numerical, 02_IEA                )  hit=True   retrieved=02_IEA_0077;02_IEA_0014;02_IEA_0086
  Q05  (   factoid, 02_IEA                )  hit=True   retrieved=02_IEA_0027;02_IEA_0028;02_IEA_0029
  Q06  ( synthesis, 02_IEA                )  hit=True   retrieved=02_IEA_0315;02_IEA_0134;02_IEA_0016
  Q07  (   factoid, 03_Google             )  hit=False  retrieved=02_IEA_0486;03_Google_0058;03_Google_0082
  Q08  (   factoid, 03_Google             )  hit=True   retrieved=03_Google_0003;03_Google_0015;03_Google_0014
  Q09  ( synthesis, 03_Google             )  hit=False  retrieved=03_Google_0079;03_Google_0058;03_Goo

In [8]:
# Cell 9 — sanity asserts.
#
# Schema lock + value-level checks. These mirror the assert pattern from
# 02_measurement_layer.ipynb cell 10 (Phase C decision: schema lock-in
# precedes value asserts so a column-name typo gives a clean diff).

# (i) Schema lock — column names AND order match §11 exactly.
on_disk = pd.read_csv(RECALL_LOG_PATH)
assert_recall_schema(on_disk)
print("S1 PASS: recall_log.csv columns match §11 schema by name and order.")

# (ii) Row count matches the locked dedup count.
assert len(on_disk) == EXPECTED_DEDUP_CELL_COUNT, (
    f"S2 FAIL: expected {EXPECTED_DEDUP_CELL_COUNT} rows, got {len(on_disk)}"
)
print(f"S2 PASS: row count == {EXPECTED_DEDUP_CELL_COUNT} (locked dedup count).")

# (iii) k values restricted to {1, 3, 5}.
got_k = set(int(k) for k in on_disk["top_k"].unique())
assert got_k <= EXPECTED_K_VALUES, (
    f"S3 FAIL: top_k contains values outside {EXPECTED_K_VALUES}: {got_k}"
)
print(f"S3 PASS: top_k values are subset of {EXPECTED_K_VALUES} (got {sorted(got_k)}).")

# (iv) strict_hit / lenient_hit non-empty exactly when question_id == Q13.
# Note: pandas read_csv coerces empty strings to NaN — use .fillna("") before
# string compares so the empty-vs-set distinction is preserved through the
# round-trip, not aliased to the literal "nan".
is_q13 = on_disk["question_id"] == Q13_ID
non_q13_strict = on_disk.loc[~is_q13, "strict_hit"].fillna("").astype(str).str.strip()
non_q13_lenient = on_disk.loc[~is_q13, "lenient_hit"].fillna("").astype(str).str.strip()
assert (non_q13_strict == "").all(), "S4 FAIL: strict_hit set on a non-Q13 row."
assert (non_q13_lenient == "").all(), "S4 FAIL: lenient_hit set on a non-Q13 row."
q13_strict = on_disk.loc[is_q13, "strict_hit"].fillna("").astype(str).str.strip()
q13_lenient = on_disk.loc[is_q13, "lenient_hit"].fillna("").astype(str).str.strip()
assert q13_strict.isin({"True", "False"}).all(), "S4 FAIL: bad strict_hit value on Q13."
assert q13_lenient.isin({"True", "False"}).all(), "S4 FAIL: bad lenient_hit value on Q13."
print("S4 PASS: strict_hit / lenient_hit set iff question_id == Q13.")

# (v) For Q13 rows, hit equals lenient_hit (cell-1 rule: aggregate uses
#     lenient definition; strict-vs-lenient gap is reported separately).
q13_hit_bool = on_disk.loc[is_q13, "hit"].astype(bool).values
q13_lenient_bool = (q13_lenient == "True").values
assert (q13_hit_bool == q13_lenient_bool).all(), (
    "S5 FAIL: Q13 rows have hit != lenient_hit."
)
print("S5 PASS: Q13 hit == lenient_hit on every Q13 row.")

# (vi) Every row's question_id is in {Q01..Q13}.
expected_qids = {f"Q{n:02d}" for n in range(1, 14)}
got_qids = set(on_disk["question_id"].unique())
assert got_qids == expected_qids, (
    f"S6 FAIL: question_ids drifted. expected={sorted(expected_qids)}, got={sorted(got_qids)}"
)
print("S6 PASS: every question_id is in Q01..Q13.")

# (vii) gold_chunk_ids and retrieved_chunk_ids use the inherited LIST_SEP=';'.
for col in ("gold_chunk_ids", "retrieved_chunk_ids"):
    bad = on_disk[on_disk[col].astype(str).str.contains(",", na=False)]
    assert bad.empty, f"S7 FAIL: comma found in {col} (expected '{LIST_SEP}')."
print(f"S7 PASS: gold/retrieved chunk_id lists use '{LIST_SEP}' separator.")

# (viii) hit is exactly True when retrieved overlaps gold_chunk_ids (non-Q13).
for _, r in on_disk[~is_q13].iterrows():
    gold_set = set(s for s in str(r["gold_chunk_ids"]).split(LIST_SEP) if s)
    ret_set  = set(s for s in str(r["retrieved_chunk_ids"]).split(LIST_SEP) if s)
    expected = bool(gold_set & ret_set) if gold_set else False
    assert bool(r["hit"]) == expected, (
        f"S8 FAIL: hit mismatch on {r['config_hash']}/{r['top_k']}/{r['question_id']}"
    )
print("S8 PASS: hit is consistent with gold ∩ retrieved on every non-Q13 row.")

# (ix) Every (config_hash, top_k) cell has exactly 13 rows (one per question).
cell_counts = on_disk.groupby(["config_hash", "top_k"]).size().rename("n_q").reset_index()
bad_cells = cell_counts[cell_counts["n_q"] != 13]
assert bad_cells.empty, f"S9 FAIL: cells without 13 questions:\n{bad_cells}"
print(f"S9 PASS: every (config_hash, top_k) cell has exactly 13 questions ({len(cell_counts)} cells).")

print("\nAll sanity asserts PASSED.")


S1 PASS: recall_log.csv columns match §11 schema by name and order.
S2 PASS: row count == 156 (locked dedup count).
S3 PASS: top_k values are subset of {1, 3, 5} (got [1, 3, 5]).
S4 PASS: strict_hit / lenient_hit set iff question_id == Q13.
S5 PASS: Q13 hit == lenient_hit on every Q13 row.
S6 PASS: every question_id is in Q01..Q13.
S7 PASS: gold/retrieved chunk_id lists use ';' separator.
S8 PASS: hit is consistent with gold ∩ retrieved on every non-Q13 row.
S9 PASS: every (config_hash, top_k) cell has exactly 13 questions (12 cells).

All sanity asserts PASSED.


In [9]:
# Cell 10 — wrap-up.
summary = {
    "recall_log_rows": len(recall_df),
    "recall_log_path": str(RECALL_LOG_PATH.relative_to(PROJECT_ROOT)),
    "unique_configs_x_topk_cells": int(
        recall_df.groupby(["config_hash", "top_k"]).ngroups
    ),
    "k_values_present": sorted(int(k) for k in recall_df["top_k"].unique()),
    "winner_recall_at_3": float(
        recall_df[(recall_df.config_hash == WINNER_CFG) & (recall_df.top_k == WINNER_K)]["hit"].mean()
    ),
    "q13_strict_hits": int(q13_summary["strict_hit"].sum()),
    "q13_lenient_hits": int(q13_summary["lenient_hit"].sum()),
    "q13_total": len(q13_summary),
}
for k, v in summary.items():
    print(f"  {k}: {v}")
print("\nBatch A — done. Phase E remains in progress (E.3 grading + E.4 plots are separate batches).")


  recall_log_rows: 156
  recall_log_path: results/recall_log.csv
  unique_configs_x_topk_cells: 12
  k_values_present: [1, 3, 5]
  winner_recall_at_3: 0.6923076923076923
  q13_strict_hits: 0
  q13_lenient_hits: 2
  q13_total: 12

Batch A — done. Phase E remains in progress (E.3 grading + E.4 plots are separate batches).
